# Chapter 44:  ChromaDB Fundamentals — collections and vector storage

In Chapter 42, you stored embeddings in ChromaDB. In Chapter 43, you compared vector databases with SQL.
Now we'll go deeper into how to manage records inside ChromaDB: create collections, add documents, retrieve records, update existing data, and delete records.
This notebook is fully self-contained. It creates its own sample data, initializes its own embedding model, and uses an in-memory database. It does not depend on any previous chapter or external data files

# Step 1: Install the dependencies 

In [ ]:
%pip install chromadb sentence-transformers numpy

# Step 2 — Import the libraries

In [1]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Step 3 — Create the dataset

In [2]:
documents = [
    "Banks monitor international payments for suspicious activity.",
    "Money laundering conceals the illegal origin of funds.",
    "Customer due diligence verifies customer identities.",
    "Credit risk models estimate the probability of loan default.",
    "Machine learning detects unusual transaction patterns."
]

metadatas = [
    {"category": "AML", "status": "active"},
    {"category": "AML", "status": "active"},
    {"category": "KYC", "status": "active"},
    {"category": "Risk", "status": "active"},
    {"category": "AI", "status": "active"}
]

In [3]:
ids = [f"doc_{i}" for i in range(1, len(documents) + 1)]


In [5]:
ids

['doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5']

We use three separate pieces of information:
- IDs: uniquely identify records.
- Documents: hold the actual text.
- Metadata: stores structured attributes that can be filtered later.

# Step 4 — Generate embeddings
ChromaDB can store vectors supplied by an external embedding model. We'll generate the vectors ourselves to make the process explicit.

In [6]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [13]:
embeddings = model.encode(documents)

In [14]:
embeddings

array([[ 0.02980763, -0.04411102, -0.06704751, ..., -0.09322759,
         0.0378824 , -0.09450466],
       [-0.02550222,  0.01433703, -0.0452748 , ..., -0.02897054,
         0.15225162, -0.02563389],
       [-0.10273864,  0.04247324,  0.05913257, ...,  0.04048206,
         0.06826895, -0.08979785],
       [ 0.01108949, -0.08210703, -0.02975351, ...,  0.01017951,
        -0.03442628, -0.07196967],
       [-0.0320264 , -0.03995375,  0.01447618, ..., -0.01906533,
         0.01269929, -0.03884009]], dtype=float32)

In [15]:
embeddings.shape

(5, 384)

# Step 5 - Create an in memory ChromaDB client

In [17]:
client = chromadb.EphemeralClient()

An ephemeral client keeps the database in memory. This is ideal for experimenting without depending on saved databases from other chapters.

# Step 6 — Create a collection

In [18]:
collection = client.create_collection(
    name="chapter_44_records",
    metadata={"hnsw:space": "cosine"}
)

In [19]:
collection.name

'chapter_44_records'

In [21]:
collection.count()

0

A collection is a container for records that includes their IDs, documents, metadata, and embeddings.

# Step 7 - Create records using add()

In [22]:
collection.add(
    ids = ids, 
    documents=documents,
    metadatas=metadatas, 
    embeddings=embeddings

)

In [23]:
collection.count()

5

# Step 8 — Read records using get()

In [24]:
record = collection.get(
    ids=["doc_1"],
    include=["documents", "metadatas", "embeddings"]
)


In [27]:
record

{'ids': ['doc_1'],
 'embeddings': array([[ 2.98076291e-02, -4.41110171e-02, -6.70475066e-02,
         -5.47525808e-02,  4.51406389e-02, -8.16947222e-02,
          1.24392487e-01, -6.21990189e-02,  8.32085386e-02,
          1.43644575e-03, -4.67694923e-03, -8.08523409e-03,
         -9.42029152e-03,  6.71868846e-02, -8.81922916e-02,
         -7.27461651e-02, -2.34822668e-02, -2.77816541e-02,
          1.41124455e-02,  2.54390370e-02, -2.54847687e-02,
         -8.46901312e-02,  6.98584539e-04,  1.68808997e-02,
         -1.32075315e-02, -2.38295160e-02,  6.46908826e-04,
         -2.68502696e-03, -3.73915136e-02, -7.97673408e-03,
          1.81226712e-03,  3.75949219e-02, -1.25874551e-02,
         -2.17443220e-02,  2.89165676e-02, -6.99069872e-02,
          8.10194090e-02, -9.28811170e-03,  3.28104571e-02,
         -3.85061614e-02,  5.99840917e-02, -9.44331437e-02,
          1.18140183e-01, -2.18085982e-02, -1.08526861e-02,
          4.69324440e-02,  1.51479868e-02,  5.95746115e-02,
       

In [28]:
aml_records = collection.get(
    where={"category": "AML"},
    include=["documents", "metadatas"]
)

In [29]:
aml_records

{'ids': ['doc_1', 'doc_2'],
 'embeddings': None,
 'documents': ['Banks monitor international payments for suspicious activity.',
  'Money laundering conceals the illegal origin of funds.'],
 'uris': None,
 'included': ['documents', 'metadatas'],
 'data': None,
 'metadatas': [{'status': 'active', 'category': 'AML'},
  {'status': 'active', 'category': 'AML'}]}

# Step 9 : Use upsert() to insert or update

In [ ]:
new_document = "Fraud detection systems identify potentially fraudulent financial behavior."
new_embedding = model.encode([new_document])


In [33]:
collection.upsert(
    ids=["doc_6"],
    documents=[new_document],
    metadatas=[
        {"category": "Fraud", "status": "active"}
    ],
    embeddings=new_embedding
)


In [34]:
collection.count()

6

In [35]:
collection.get(ids = ["doc_6"])

{'ids': ['doc_6'],
 'embeddings': None,
 'documents': ['Fraud detection systems identify potentially fraudulent financial behavior.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'category': 'Fraud', 'status': 'active'}]}

# Step 12 - Perform Semantic Search

In [36]:
query = "How can banks identify suspicious international transfers?"


In [37]:
query_embedding = model.encode([query])


In [38]:
result = collection.query(query_embeddings=query_embedding, n_results=3)

In [39]:
result

{'ids': [['doc_1', 'doc_2', 'doc_6']],
 'embeddings': None,
 'documents': [['Banks monitor international payments for suspicious activity.',
   'Money laundering conceals the illegal origin of funds.',
   'Fraud detection systems identify potentially fraudulent financial behavior.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'status': 'active', 'category': 'AML'},
   {'status': 'active', 'category': 'AML'},
   {'status': 'active', 'category': 'Fraud'}]],
 'distances': [[0.1386709213256836, 0.4859205484390259, 0.5346649885177612]]}